# 1.4 Testing with Pytest> ☕ **Java parallel:** Pytest is to Python what JUnit + Mockito is to Java — but simpler.---

## Pytest vs JUnit at a Glance| JUnit | Pytest ||-------|--------|| `@Test` annotation | Functions starting with `test_` || `assertEquals(expected, actual)` | `assert actual == expected` || `@BeforeEach` | `@pytest.fixture` || `@ParameterizedTest` | `@pytest.mark.parametrize` || Mockito `mock()` | `unittest.mock.patch` |

In [ ]:
# %%writefile test_pipeline.py# (In practice, save this as a .py file and run with: pytest test_pipeline.py -v)import pytestfrom dataclasses import dataclass@dataclassclass Record:    id: int    value: float    is_valid: bool = Truedef validate_records(records: list[Record]) -> list[Record]:    """Filter out invalid records and those with negative values."""    return [r for r in records if r.is_valid and r.value >= 0]# --- Tests ---def test_validate_removes_invalid():    records = [Record(1, 10.0), Record(2, 20.0, is_valid=False)]    result = validate_records(records)    assert len(result) == 1    assert result[0].id == 1def test_validate_removes_negative():    records = [Record(1, -5.0), Record(2, 10.0)]    result = validate_records(records)    assert len(result) == 1def test_validate_empty_list():    assert validate_records([]) == []# Parametrized test — like @ParameterizedTest in JUnit@pytest.mark.parametrize("value,expected_valid", [    (100.0, True),    (0.0, True),    (-1.0, False),    (-0.01, False),])def test_value_boundary(value, expected_valid):    record = Record(id=1, value=value)    result = validate_records([record])    assert (len(result) == 1) == expected_valid

## Fixtures — Setup & Teardown☕ **Java parallel:** `@BeforeEach` / `@AfterEach`, but composable and reusable.

In [ ]:
import pytest# Fixture — creates test data (like @BeforeEach)@pytest.fixturedef sample_records():    return [        Record(1, 100.0),        Record(2, 200.0),        Record(3, -50.0, is_valid=False),        Record(4, 0.0),    ]# Fixture with teardown (like @AfterEach)@pytest.fixturedef temp_database():    db = {"connected": True, "data": []}  # setup    yield db                                # test runs here    db["connected"] = False                 # teardown    print("DB cleaned up")def test_with_sample_data(sample_records):    """Fixtures are injected by name — like Spring DI!"""    valid = validate_records(sample_records)    assert len(valid) == 2def test_with_db(temp_database):    temp_database["data"].append({"id": 1})    assert len(temp_database["data"]) == 1

## Running Tests```bash# Run all testspytest# Verbose outputpytest -v# Run specific testpytest test_pipeline.py::test_validate_empty_list# Run with coveragepip install pytest-covpytest --cov=. --cov-report=html# Stop on first failurepytest -x```